In [ ]:
"""
Bank Selection Bias Experiment (N=500, Y-dependent MNAR)
- High-income middle-aged+ customers hide loan demand
- Models: LR, RF, MLP, TabPFN, FT-TabPFN
- Output: ΔAccuracy, ΔDP, ΔEO + Sample Size Control
"""

import pandas as pd
import numpy as np
import torch
from torch.optim import Adam
from torch.utils.data import DataLoader
from functools import partial
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
import fairlearn.metrics as fm
from tabpfn import TabPFNClassifier
from tabpfn.utils import meta_dataset_collator
from tabpfn.finetune_utils import clone_model_for_evaluation
import warnings

# Suppress warnings for cleaner output during execution
warnings.filterwarnings("ignore")


# ==================== SAFE UTILS ====================
def to_numpy_safe(x):
    """
    Safely convert input to numpy array, handling torch Tensors.

    Args:
        x: Input tensor or array.

    Returns:
        numpy.ndarray: Converted array.
    """
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.array(x)


# ==================== SELECTION BIAS (BANK) ====================
def simulate_selection_bias_bank(df, n_samples=500, del_prob_high=0.75, del_prob_low=0.1, seed=42):
    """
    Simulate MNAR selection bias in the bank dataset by deleting high-risk samples.

    High-risk group: middle-aged and elderly + loan + high income + high credit card spending.

    Args:
        df (pd.DataFrame): Original dataset.
        n_samples (int): Initial sample size before deletion.
        del_prob_high (float): Probability of deleting high-risk samples.
        del_prob_low (float): Probability of deleting low-risk samples.
        seed (int): Random seed for reproducibility.

    Returns:
        tuple: (df_biased: biased dataset, df_small: initial sampled dataset)
    """
    np.random.seed(seed)
    df = df.copy()

    # Define high-risk group
    high_risk = (
        (df['Age'] > 50) &
        (df['Personal Loan'] == 1) &
        (df['Income'] > 100) &
        (df['CCAvg'] > 3)
    )

    # Stratified sampling to maintain class balance
    pos = df[df['Personal Loan'] == 1].sample(frac=1, random_state=seed)
    neg = df[df['Personal Loan'] == 0].sample(frac=1, random_state=seed)
    n_pos = min(int(n_samples * df['Personal Loan'].mean()), len(pos))
    n_neg = n_samples - n_pos
    df_small = pd.concat([pos.iloc[:n_pos], neg.iloc[:n_neg]]).sample(frac=1, random_state=seed).reset_index(drop=True)

    # Apply MNAR deletion based on risk
    keep = np.ones(len(df_small), dtype=bool)
    for i in range(len(df_small)):
        if high_risk.iloc[df_small.index[i]]:
            keep[i] = np.random.rand() > del_prob_high
        else:
            keep[i] = np.random.rand() > del_prob_low

    df_biased = df_small[keep].reset_index(drop=True)
    print(f"[Bias] N={len(df_small)} to {len(df_biased)} (ΔN={len(df_small)-len(df_biased)})")
    return df_biased, df_small


# ==================== PREPROCESS ====================
def preprocess_dataset(df, target, drop_cols=None):
    """
    Preprocess the dataset: drop columns, separate features/target, apply scaling and encoding.

    Args:
        df (pd.DataFrame): Input dataset.
        target (str): Target column name.
        drop_cols (list): Columns to drop.

    Returns:
        tuple: (X_proc: processed features, y: target, X_raw: raw features, preprocessor: fitted transformer)
    """
    if drop_cols:
        df = df.drop(columns=drop_cols, errors='ignore')

    X = df.drop(columns=[target])
    y = df[target].values
    num_cols = X.select_dtypes('number').columns.tolist()
    cat_cols = X.select_dtypes('object').columns.tolist()

    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ])
    X_proc = preprocessor.fit_transform(X)

    return X_proc, y, X, preprocessor


# ==================== EVALUATE ====================
def evaluate_model(model, X_train, y_train, X_test, y_test, sensitive,
                   is_tabpfn=False, preprocessor=None, eval_cfg=None):
    """
    Evaluate a model on test data, computing accuracy, DP, and EO.

    Args:
        model: The model to evaluate.
        X_train: Training features (raw for TabPFN, processed otherwise).
        y_train: Training targets.
        X_test: Test features (raw).
        y_test: Test targets.
        sensitive: Sensitive feature values for test set.
        is_tabpfn (bool): If True, handle TabPFN-specific fitting.
        preprocessor: Fitted preprocessor for non-TabPFN models.
        eval_cfg (dict): Config for TabPFN evaluation.

    Returns:
        tuple: (accuracy, demographic_parity_difference, equalized_odds_difference)
    """
    try:
        if is_tabpfn:
            if not hasattr(model, 'softmax_temperature_'):
                model.fit(X_train[:10], y_train[:10])
            eval_model = clone_model_for_evaluation(model, eval_cfg or {}, TabPFNClassifier)
            eval_model.fit(X_train, y_train)
            preds = eval_model.predict(X_test)
        else:
            X_train_pp = preprocessor.transform(X_train)
            X_test_pp = preprocessor.transform(X_test)
            model.fit(X_train_pp, y_train)
            preds = model.predict(X_test_pp)

        acc = accuracy_score(y_test, preds)
        dp = fm.demographic_parity_difference(y_test, preds, sensitive_features=sensitive)
        eo = fm.equalized_odds_difference(y_test, preds, sensitive_features=sensitive)
        return acc, dp, eo
    except Exception as e:
        print(f"[Eval Error] {e}")
        return 0.0, 0.0, 0.0


# ==================== MAIN ====================
def main(data_path='/content/Bank_Personal_Loan_Modelling.xlsx'):
    """
    Main experiment runner.

    Args:
        data_path (str): Path to the Bank dataset Excel file.
    """
    config = {
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "random_seed": 42,
        "valid_set_ratio": 0.3,
        "n_inference_context_samples": 5000,
        "finetuning": {
            "epochs": 5,
            "learning_rate": 1e-5,
            "meta_batch_size": 1,
            "batch_size": 128
        }
    }

    # Load Bank dataset
    df = pd.read_excel(data_path, sheet_name='Data')
    print("Bank missing values:\n", df.isnull().sum())

    results = []
    for run in range(5):
        seed = 42 + run
        config["random_seed"] = seed
        np.random.seed(seed)
        torch.manual_seed(seed)
        print(f"\n{'='*20} RUN {run+1} {'='*20}")

        # Simulate bias
        df_biased, df_clean = simulate_selection_bias_bank(df, n_samples=500, seed=seed)
        if len(df_biased) < 100:
            print("Too few samples, skip.")
            continue

        # Sample size control: reduce clean data to match biased size
        df_clean_reduced = df_clean.sample(n=len(df_biased), random_state=seed)
        print(f"[Control] Clean_Reduced: {len(df_clean_reduced)} samples")

        # Preprocess datasets
        drop_cols = ["ID", "ZIP Code"]
        Xb_proc, yb, Xb_raw, preproc_b = preprocess_dataset(df_biased, "Personal Loan", drop_cols)
        Xc_proc, yc, Xc_raw, _ = preprocess_dataset(df_clean, "Personal Loan", drop_cols)
        Xr_proc, yr, Xr_raw, _ = preprocess_dataset(df_clean_reduced, "Personal Loan", drop_cols)

        # Split datasets (train/test, no stratification for simplicity)
        splitter = partial(train_test_split, test_size=0.3, random_state=seed, stratify=None)
        Xb_train_raw, Xb_test_raw, yb_train, yb_test = splitter(Xb_raw, yb)
        _, Xc_test_raw, _, yc_test = splitter(Xc_raw, yc)
        Xr_train_raw, Xr_test_raw, yr_train, yr_test = splitter(Xr_raw, yr)

        # Extract sensitive features for test sets (using Education as proxy)
        s_test_b = df_biased.loc[Xb_test_raw.index, "Education"]
        s_test_c = df_clean.loc[Xc_test_raw.index, "Education"]
        s_test_r = df_clean_reduced.loc[Xr_test_raw.index, "Education"]

        # Define models
        models = {
            'LR': Pipeline([('clf', LogisticRegression(max_iter=1000, C=0.1))]),
            'RF': Pipeline([('clf', RandomForestClassifier(max_depth=5, n_estimators=50))]),
            'MLP': Pipeline([('clf', MLPClassifier(max_iter=300, hidden_layer_sizes=(50,), alpha=0.01))]),
            'TabPFN': TabPFNClassifier(device=config["device"]),
            'FT-TabPFN': None
        }

        # Finetune FT-TabPFN
        print("Finetuning FT-TabPFN...")
        try:
            classifier_cfg = {
                "ignore_pretraining_limits": True,
                "device": config["device"],
                "n_estimators": 1,
                "random_state": config["random_seed"],
                "inference_precision": torch.float32,
            }
            clf = TabPFNClassifier(**classifier_cfg, fit_mode="batched", differentiable_input=False)
            clf._initialize_model_variables()

            # Prepare dummy data for initial fit
            dummy_X = Xb_train_raw.values[:16]
            dummy_y = yb_train[:16]
            if len(np.unique(dummy_y)) < 2:
                pos_idx = np.where(yb_train == 1)[0][0]
                neg_idx = np.where(yb_train == 0)[0][0]
                dummy_X = np.vstack([dummy_X, Xb_train_raw.values[pos_idx:pos_idx+1], Xb_train_raw.values[neg_idx:neg_idx+1]])
                dummy_y = np.hstack([dummy_y, [1, 0]])

            clf.fit(dummy_X, dummy_y)

            # Prepare training datasets for finetuning
            train_splitter = partial(train_test_split, test_size=config["valid_set_ratio"], random_state=seed)
            training_datasets = clf.get_preprocessed_datasets(
                Xb_train_raw.values, yb_train,
                train_splitter,
                config["finetuning"]["batch_size"]
            )
            loader = DataLoader(training_datasets, batch_size=1, collate_fn=meta_dataset_collator, shuffle=True)
            optimizer = Adam(clf.models_[0].parameters(), lr=config["finetuning"]["learning_rate"])
            loss_fn = torch.nn.CrossEntropyLoss()

            # Finetuning loop
            for epoch in range(config["finetuning"]["epochs"]):
                epoch_loss = 0.0
                n_batch = 0
                pbar = tqdm(loader, desc=f"Epoch {epoch+1}", leave=False)
                for (X_tr, X_val, y_tr, y_val, cat, conf) in pbar:
                    if len(np.unique(to_numpy_safe(y_tr))) < 2 or len(np.unique(to_numpy_safe(y_val))) < 2:
                        continue
                    optimizer.zero_grad()
                    clf.fit_from_preprocessed(X_tr, y_tr, cat, conf)
                    logits = clf.forward(X_val, return_logits=True)
                    loss = loss_fn(logits, y_val.to(config["device"]))
                    loss.backward()
                    optimizer.step()
                    epoch_loss += loss.item()
                    n_batch += 1
                    pbar.set_postfix(loss=f"{loss.item():.4f}")
                if n_batch > 0:
                    print(f" -> Epoch {epoch+1}, Avg Loss: {epoch_loss/n_batch:.5f}")

            models['FT-TabPFN'] = clf
        except Exception as e:
            print(f"[FT Error] {e}")
            models['FT-TabPFN'] = None

        # Evaluation setup
        eval_cfg = {**classifier_cfg, "inference_config": {"SUBSAMPLE_SAMPLES": config["n_inference_context_samples"]}}

        # Evaluate each model
        for name, model in models.items():
            if model is None:
                continue

            # 1. Biased train vs Clean full test
            acc_b, dp_b, eo_b = evaluate_model(
                model, Xb_train_raw, yb_train, Xc_test_raw, yc_test, s_test_c,
                is_tabpfn=name in ['TabPFN', 'FT-TabPFN'], preprocessor=preproc_b, eval_cfg=eval_cfg
            )
            acc_c, dp_c, eo_c = evaluate_model(
                model, Xc_raw, yc, Xc_test_raw, yc_test, s_test_c,
                is_tabpfn=name in ['TabPFN', 'FT-TabPFN'], preprocessor=preproc_b, eval_cfg=eval_cfg
            )

            # 2. Biased train vs Clean reduced test
            acc_r, dp_r, eo_r = evaluate_model(
                model, Xr_train_raw, yr_train, Xc_test_raw, yc_test, s_test_c,
                is_tabpfn=name in ['TabPFN', 'FT-TabPFN'], preprocessor=preproc_b, eval_cfg=eval_cfg
            )

            results.append({
                'Run': run+1, 'Model': name, 'Sensitive': 'Education',
                'Acc_B': acc_b, 'Acc_C': acc_c, 'Acc_R': acc_r,
                'DP_B': dp_b, 'DP_C': dp_c, 'DP_R': dp_r,
                'EO_B': eo_b, 'EO_C': eo_c, 'EO_R': eo_r,
                'ΔAcc_vs_C': acc_b - acc_c, 'ΔDP_vs_C': dp_b - dp_c, 'ΔEO_vs_C': eo_b - eo_c,
                'ΔAcc_vs_R': acc_b - acc_r, 'ΔDP_vs_R': dp_b - dp_r, 'ΔEO_vs_R': eo_b - eo_r,
            })

    # Aggregate and print results
    if not results:
        print("No results!")
        return

    df_res = pd.DataFrame(results)
    agg = df_res.groupby('Model').agg({
        'ΔAcc_vs_C': ['mean', 'std'], 'ΔDP_vs_C': ['mean', 'std'], 'ΔEO_vs_C': ['mean', 'std'],
        'ΔAcc_vs_R': ['mean', 'std'], 'ΔDP_vs_R': ['mean', 'std'], 'ΔEO_vs_R': ['mean', 'std'],
        'Acc_B': 'mean', 'DP_B': 'mean', 'EO_B': 'mean'
    }).round(4)

    agg.columns = ['ΔAcc_C_mean', 'ΔAcc_C_std', 'ΔDP_C_mean', 'ΔDP_C_std', 'ΔEO_C_mean', 'ΔEO_C_std',
                   'ΔAcc_R_mean', 'ΔAcc_R_std', 'ΔDP_R_mean', 'ΔDP_R_std', 'ΔEO_R_mean', 'ΔEO_R_std',
                   'Acc_Biased', 'DP_Biased', 'EO_Biased']
    agg = agg[['Acc_Biased', 'DP_Biased', 'EO_Biased',
               'ΔAcc_C_mean', 'ΔDP_C_mean', 'ΔEO_C_mean',
               'ΔAcc_R_mean', 'ΔDP_R_mean', 'ΔEO_R_mean']]

    print("\n" + "="*100)
    print("FINAL RESULTS: Bank Selection Bias + Sample Size Control")
    print("="*100)
    print(agg.to_markdown(index=True))
    print("\nBest Δ (lowest abs):")
    print("vs Clean_Full:", agg[['ΔDP_C_mean', 'ΔEO_C_mean']].abs().idxmin())
    print("vs Clean_Reduced:", agg[['ΔDP_R_mean', 'ΔEO_R_mean']].abs().idxmin())


# ==================== RUN ====================
if __name__ == "__main__":
    # Update data_path as needed for your environment
    main(data_path='Bank_Personal_Loan_Modelling.xlsx')

In [ ]:
"""
Adult Selection Bias Experiment (N=500, Y-dependent MNAR)
- High-income working females (>50 hours/week) hide income data
- Models: LR, RF, MLP, TabPFN, FT-TabPFN
- Output: ΔAccuracy, ΔDP, ΔEO + Sample Size Control
"""

import pandas as pd
import numpy as np
import torch
from torch.optim import Adam
from torch.utils.data import DataLoader
from functools import partial
from tqdm import tqdm
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.metrics import accuracy_score
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
import fairlearn.metrics as fm
from tabpfn import TabPFNClassifier
from tabpfn.utils import meta_dataset_collator
from tabpfn.finetune_utils import clone_model_for_evaluation
import warnings

# Suppress warnings for cleaner output during execution
warnings.filterwarnings("ignore")


# ==================== SAFE UTILS ====================
def to_numpy_safe(x):
    """
    Safely convert input to numpy array, handling torch Tensors.

    Args:
        x: Input tensor or array.

    Returns:
        numpy.ndarray: Converted array.
    """
    if isinstance(x, torch.Tensor):
        return x.detach().cpu().numpy()
    return np.array(x)


# ==================== SELECTION BIAS (ADULT) ====================
def simulate_selection_bias_adult(df, n_samples=500, del_prob_high=0.7, del_prob_low=0.1, seed=42):
    """
    Simulate MNAR selection bias in the Adult dataset by deleting high-risk samples.

    High-risk group: female + high income (income>50K=1) + high working hours (>50 hours/week).

    Args:
        df (pd.DataFrame): Original dataset.
        n_samples (int): Initial sample size before deletion.
        del_prob_high (float): Probability of deleting high-risk samples.
        del_prob_low (float): Probability of deleting low-risk samples.
        seed (int): Random seed for reproducibility.

    Returns:
        tuple: (df_biased: biased dataset, df_small: initial sampled dataset)
    """
    np.random.seed(seed)
    df = df.copy()

    # Define high-risk group
    high_risk = (
        (df['sex'] == 'Female') &
        (df['income>50K'] == 1) &
        (df['hours-per-week'] > 50)
    )

    # Stratified sampling to maintain class balance
    pos = df[df['income>50K'] == 1].sample(frac=1, random_state=seed)
    neg = df[df['income>50K'] == 0].sample(frac=1, random_state=seed)
    n_pos = min(int(n_samples * df['income>50K'].mean()), len(pos))
    n_neg = n_samples - n_pos
    df_small = pd.concat([pos.iloc[:n_pos], neg.iloc[:n_neg]]).sample(frac=1, random_state=seed).reset_index(drop=True)

    # Apply MNAR deletion based on risk
    keep = np.ones(len(df_small), dtype=bool)
    for i in range(len(df_small)):
        if high_risk.iloc[df_small.index[i]]:
            keep[i] = np.random.rand() > del_prob_high
        else:
            keep[i] = np.random.rand() > del_prob_low

    df_biased = df_small[keep].reset_index(drop=True)
    print(f"[Bias] N={len(df_small)} to {len(df_biased)} (ΔN={len(df_small)-len(df_biased)})")
    return df_biased, df_small


# ==================== PREPROCESS ====================
def preprocess_dataset(df, target, drop_cols=None):
    """
    Preprocess the dataset: drop columns, separate features/target, apply scaling and encoding.

    Args:
        df (pd.DataFrame): Input dataset.
        target (str): Target column name.
        drop_cols (list): Columns to drop.

    Returns:
        tuple: (X_proc: processed features, y: target, X_raw: raw features, preprocessor: fitted transformer)
    """
    if drop_cols:
        df = df.drop(columns=drop_cols, errors='ignore')

    X = df.drop(columns=[target])
    y = df[target].values
    num_cols = X.select_dtypes('number').columns.tolist()
    cat_cols = X.select_dtypes('object').columns.tolist()

    preprocessor = ColumnTransformer([
        ('num', StandardScaler(), num_cols),
        ('cat', OneHotEncoder(drop='first', sparse_output=False, handle_unknown='ignore'), cat_cols)
    ])
    X_proc = preprocessor.fit_transform(X)

    return X_proc, y, X, preprocessor


# ==================== EVALUATE ====================
def evaluate_model(model, X_train, y_train, X_test, y_test, sensitive,
                   is_tabpfn=False, preprocessor=None, eval_cfg=None):
    """
    Evaluate a model on test data, computing accuracy, DP, and EO.

    Args:
        model: The model to evaluate.
        X_train: Training features (raw for TabPFN, processed otherwise).
        y_train: Training targets.
        X_test: Test features (raw).
        y_test: Test targets.
        sensitive: Sensitive feature values for test set.
        is_tabpfn (bool): If True, handle TabPFN-specific fitting.
        preprocessor: Fitted preprocessor for non-TabPFN models.
        eval_cfg (dict): Config for TabPFN evaluation.

    Returns:
        tuple: (accuracy, demographic_parity_difference, equalized_odds_difference)
    """
    try:
        if is_tabpfn:
            if not hasattr(model, 'softmax_temperature_'):
                model.fit(X_train[:10], y_train[:10])
            eval_model = clone_model_for_evaluation(model, eval_cfg or {}, TabPFNClassifier)
            eval_model.fit(X_train, y_train)
            preds = eval_model.predict(X_test)
        else:
            X_train_pp = preprocessor.transform(X_train)
            X_test_pp = preprocessor.transform(X_test)
            model.fit(X_train_pp, y_train)
            preds = model.predict(X_test_pp)

        acc = accuracy_score(y_test, preds)
        dp = fm.demographic_parity_difference(y_test, preds, sensitive_features=sensitive)
        eo = fm.equalized_odds_difference(y_test, preds, sensitive_features=sensitive)
        return acc, dp, eo
    except Exception as e:
        print(f"[Eval Error] {e}")
        return 0.0, 0.0, 0.0


# ==================== MAIN ====================
def main(data_path='/content/adult.csv'):
    """
    Main experiment runner.

    Args:
        data_path (str): Path to the Adult dataset CSV file.
    """
    config = {
        "device": "cuda" if torch.cuda.is_available() else "cpu",
        "random_seed": 42,
        "valid_set_ratio": 0.3,
        "n_inference_context_samples": 5000,
        "finetuning": {
            "epochs": 5,
            "learning_rate": 1e-5,
            "meta_batch_size": 1,
            "batch_size": 128
        }
    }

    # Load Adult dataset
    df = pd.read_csv(data_path)
    print("Adult missing values:\n", df.isnull().sum())

    results = []
    for run in range(5):
        seed = 42 + run
        config["random_seed"] = seed
        np.random.seed(seed)
        torch.manual_seed(seed)
        print(f"\n{'='*20} RUN {run+1} {'='*20}")

        # Simulate bias
        df_biased, df_clean = simulate_selection_bias_adult(df, n_samples=500, seed=seed)
        if len(df_biased) < 100:
            print("Too few samples, skip.")
            continue

        # Sample size control: reduce clean data to match biased size
        df_clean_reduced = df_clean.sample(n=len(df_biased), random_state=seed)
        print(f"[Control] Clean_Reduced: {len(df_clean_reduced)} samples")

        # Preprocess datasets
        drop_cols = None  # No specific columns to drop for Adult
        Xb_proc, yb, Xb_raw, preproc_b = preprocess_dataset(df_biased, "income>50K", drop_cols)
        Xc_proc, yc, Xc_raw, _ = preprocess_dataset(df_clean, "income>50K", drop_cols)
        Xr_proc, yr, Xr_raw, _ = preprocess_dataset(df_clean_reduced, "income>50K", drop_cols)

        # Split datasets (train/test, no stratification for simplicity)
        splitter = partial(train_test_split, test_size=0.3, random_state=seed, stratify=None)
        Xb_train_raw, Xb_test_raw, yb_train, yb_test = splitter(Xb_raw, yb)
        _, Xc_test_raw, _, yc_test = splitter(Xc_raw, yc)
        Xr_train_raw, Xr_test_raw, yr_train, yr_test = splitter(Xr_raw, yr)

        # Extract sensitive features for test sets (using sex as sensitive attribute)
        s_test_c = df_clean.loc[Xc_test_raw.index, "sex"]

        # Define models
        models = {
            'LR': Pipeline([('clf', LogisticRegression(max_iter=1000, C=0.1))]),
            'RF': Pipeline([('clf', RandomForestClassifier(max_depth=5, n_estimators=50))]),
            'MLP': Pipeline([('clf', MLPClassifier(max_iter=300, hidden_layer_sizes=(50,), alpha=0.01))]),
            'TabPFN': TabPFNClassifier(device=config["device"]),
            'FT-TabPFN': None
        }

        # Finetune FT-TabPFN
        print("Finetuning FT-TabPFN...")
        try:
            classifier_cfg = {
                "ignore_pretraining_limits": True,
                "device": config["device"],
                "n_estimators": 1,
                "random_state": config["random_seed"],
                "inference_precision": torch.float32,
            }
            clf = TabPFNClassifier(**classifier_cfg, fit_mode="batched", differentiable_input=False)
            clf._initialize_model_variables()

            # Prepare dummy data for initial fit
            dummy_X = Xb_train_raw.values[:16]
            dummy_y = yb_train[:16]
            if len(np.unique(dummy_y)) < 2:
                pos_idx = np.where(yb_train == 1)[0][0]
                neg_idx = np.where(yb_train == 0)[0][0]
                dummy_X = np.vstack([dummy_X, Xb_train_raw.values[pos_idx:pos_idx+1], Xb_train_raw.values[neg_idx:neg_idx+1]])
                dummy_y = np.hstack([dummy_y, [1, 0]])

            clf.fit(dummy_X, dummy_y)

            # Prepare training datasets for finetuning
            train_splitter = partial(train_test_split, test_size=config["valid_set_ratio"], random_state=seed)
            training_datasets = clf.get_preprocessed_datasets(
                Xb_train_raw.values, yb_train,
                train_splitter,
                config["finetuning"]["batch_size"]
            )
            loader = DataLoader(training_datasets, batch_size=1, collate_fn=meta_dataset_collator, shuffle=True)
            optimizer = Adam(clf.models_[0].parameters(), lr=config["finetuning"]["learning_rate"])
            loss_fn = torch.nn.CrossEntropyLoss()

            # Finetuning loop
            for epoch in range(config["finetuning"]["epochs"]):
                epoch_loss = 0.0
                n_batch = 0
                pbar = tqdm(loader, desc=f"Epoch {epoch+1}", leave=False)
                for (X_tr, X_val, y_tr, y_val, cat, conf) in pbar:
                    if len(np.unique(to_numpy_safe(y_tr))) < 2 or len(np.unique(to_numpy_safe(y_val))) < 2:
                        continue
                    optimizer.zero_grad()
                    clf.fit_from_preprocessed(X_tr, y_tr, cat, conf)
                    logits = clf.forward(X_val, return_logits=True)
                    loss = loss_fn(logits, y_val.to(config["device"]))
                    loss.backward()
                    optimizer.step()
                    epoch_loss += loss.item()
                    n_batch += 1
                    pbar.set_postfix(loss=f"{loss.item():.4f}")
                if n_batch > 0:
                    print(f" -> Epoch {epoch+1}, Avg Loss: {epoch_loss/n_batch:.5f}")

            models['FT-TabPFN'] = clf
        except Exception as e:
            print(f"[FT Error] {e}")
            models['FT-TabPFN'] = None

        # Evaluation setup
        eval_cfg = {**classifier_cfg, "inference_config": {"SUBSAMPLE_SAMPLES": config["n_inference_context_samples"]}}

        # Evaluate each model
        for name, model in models.items():
            if model is None:
                continue

            # 1. Biased train vs Clean full test
            acc_b, dp_b, eo_b = evaluate_model(
                model, Xb_train_raw, yb_train, Xc_test_raw, yc_test, s_test_c,
                is_tabpfn=name in ['TabPFN', 'FT-TabPFN'], preprocessor=preproc_b, eval_cfg=eval_cfg
            )
            acc_c, dp_c, eo_c = evaluate_model(
                model, Xc_raw, yc, Xc_test_raw, yc_test, s_test_c,
                is_tabpfn=name in ['TabPFN', 'FT-TabPFN'], preprocessor=preproc_b, eval_cfg=eval_cfg
            )

            # 2. Biased train vs Clean reduced test
            acc_r, dp_r, eo_r = evaluate_model(
                model, Xr_train_raw, yr_train, Xc_test_raw, yc_test, s_test_c,
                is_tabpfn=name in ['TabPFN', 'FT-TabPFN'], preprocessor=preproc_b, eval_cfg=eval_cfg
            )

            results.append({
                'Run': run+1, 'Model': name, 'Sensitive': 'sex',
                'Acc_B': acc_b, 'Acc_C': acc_c, 'Acc_R': acc_r,
                'DP_B': dp_b, 'DP_C': dp_c, 'DP_R': dp_r,
                'EO_B': eo_b, 'EO_C': eo_c, 'EO_R': eo_r,
                'ΔAcc_vs_C': acc_b - acc_c, 'ΔDP_vs_C': dp_b - dp_c, 'ΔEO_vs_C': eo_b - eo_c,
                'ΔAcc_vs_R': acc_b - acc_r, 'ΔDP_vs_R': dp_b - dp_r, 'ΔEO_vs_R': eo_b - eo_r,
            })

    # Aggregate and print results
    if not results:
        print("No results!")
        return

    df_res = pd.DataFrame(results)
    agg = df_res.groupby('Model').agg({
        'ΔAcc_vs_C': ['mean', 'std'], 'ΔDP_vs_C': ['mean', 'std'], 'ΔEO_vs_C': ['mean', 'std'],
        'ΔAcc_vs_R': ['mean', 'std'], 'ΔDP_vs_R': ['mean', 'std'], 'ΔEO_vs_R': ['mean', 'std'],
        'Acc_B': 'mean', 'DP_B': 'mean', 'EO_B': 'mean'
    }).round(4)

    agg.columns = ['ΔAcc_C_mean', 'ΔAcc_C_std', 'ΔDP_C_mean', 'ΔDP_C_std', 'ΔEO_C_mean', 'ΔEO_C_std',
                   'ΔAcc_R_mean', 'ΔAcc_R_std', 'ΔDP_R_mean', 'ΔDP_R_std', 'ΔEO_R_mean', 'ΔEO_R_std',
                   'Acc_Biased', 'DP_Biased', 'EO_Biased']
    agg = agg[['Acc_Biased', 'DP_Biased', 'EO_Biased',
               'ΔAcc_C_mean', 'ΔDP_C_mean', 'ΔEO_C_mean',
               'ΔAcc_R_mean', 'ΔDP_R_mean', 'ΔEO_R_mean']]

    print("\n" + "="*100)
    print("FINAL RESULTS: Adult Selection Bias + Sample Size Control")
    print("="*100)
    print(agg.to_markdown(index=True))
    print("\nBest Δ (lowest abs):")
    print("vs Clean_Full:", agg[['ΔDP_C_mean', 'ΔEO_C_mean']].abs().idxmin())
    print("vs Clean_Reduced:", agg[['ΔDP_R_mean', 'ΔEO_R_mean']].abs().idxmin())


# ==================== RUN ====================
if __name__ == "__main__":
    # Update data_path as needed for your environment
    main(data_path='adult.csv')